# Deepfake Audio Detection - Self-Contained Google Colab Training Notebook

This notebook trains the 4 refined models (Wav2Vec2, AASIST, CQCC, CustomHybrid) locally via Cloud GPUs.

In [ ]:
!pip install -q torch torchaudio torchvision transformers librosa scikit-learn matplotlib seaborn tqdm soundfile datasets scipy accelerate huggingface_hub

In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount = True)

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Model


# ============================================================
# 1. Wav2Vec2 Detector (Self-supervised Transformer Baseline)
# ============================================================
class AttentivePooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return torch.sum(w * x, dim=1)

class Wav2Vec2SpoofDetector(nn.Module):
    def __init__(self, num_classes=2, model_name="facebook/wav2vec2-base"):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained(model_name)

        #freeze model
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        hidden = self.wav2vec.config.hidden_size
        self.pool = AttentivePooling(hidden)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x):
        if x.dim() == 3:
            x = x.squeeze(1)
        out = self.wav2vec(x).last_hidden_state
        pooled = self.pool(out)
        return self.classifier(pooled)

# ============================================================
# 2. AASIST (SOTA Graph-based Baseline)
# ============================================================

import random
from typing import Union
import numpy as np
from torch import Tensor

# Original simplistic Graph Attention/Block kept for the Custom model dependent on it
class GraphAttention(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.attn = nn.Linear(out_dim * 2, 1)

    def forward(self, x):
        h = self.fc(x)
        # Instead of allocating O(N^2 * D) tensor arrays for pairwise combinations,
        # we can decompose the linear attention matrix and use broadcasting!
        # Memory consumption goes from ~10GB on N=400 to ~2MB.
        W = self.attn.weight.squeeze()
        D = h.shape[-1]

        W_1 = W[:D]
        W_2 = W[D:]

        # Compute individual node scores: shape (B, N, 1)
        score_i = torch.matmul(h, W_1).unsqueeze(-1)
        score_j = torch.matmul(h, W_2).unsqueeze(-1)

        # Broadcast (B, N, 1) + (B, 1, N) -> (B, N, N)
        e = score_i + score_j.transpose(1, 2)

        if self.attn.bias is not None:
            e = e + self.attn.bias

        alpha = F.softmax(e, dim=-1)
        out = torch.matmul(alpha, h)
        return out

class GraphBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gat = GraphAttention(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        res = x
        x = self.gat(x)
        x = self.dropout(x)
        x = self.norm(x + res)
        return x

class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, **kwargs):
        super().__init__()

        # attention map
        self.att_proj = nn.Linear(in_dim, out_dim)
        self.att_weight = self._init_new_params(out_dim, 1)

        # project
        self.proj_with_att = nn.Linear(in_dim, out_dim)
        self.proj_without_att = nn.Linear(in_dim, out_dim)

        # batch norm
        self.bn = nn.BatchNorm1d(out_dim)

        # dropout for inputs
        self.input_drop = nn.Dropout(p=0.2)

        # activate
        self.act = nn.SELU(inplace=True)

        # temperature
        self.temp = 1.
        if "temperature" in kwargs:
            self.temp = kwargs["temperature"]

    def forward(self, x):
        '''
        x   :(#bs, #node, #dim)
        '''
        # apply input dropout
        x = self.input_drop(x)

        # derive attention map
        att_map = self._derive_att_map(x)

        # projection
        x = self._project(x, att_map)

        # apply batch norm
        x = self._apply_BN(x)
        x = self.act(x)
        return x

    def _pairwise_mul_nodes(self, x):
        '''
        Calculates pairwise multiplication of nodes.
        - for attention map
        x           :(#bs, #node, #dim)
        out_shape   :(#bs, #node, #node, #dim)
        '''

        nb_nodes = x.size(1)
        x = x.unsqueeze(2).expand(-1, -1, nb_nodes, -1)
        x_mirror = x.transpose(1, 2)

        return x * x_mirror

    def _derive_att_map(self, x):
        '''
        x           :(#bs, #node, #dim)
        out_shape   :(#bs, #node, #node, 1)
        '''
        att_map = self._pairwise_mul_nodes(x)
        # size: (#bs, #node, #node, #dim_out)
        att_map = torch.tanh(self.att_proj(att_map))
        # size: (#bs, #node, #node, 1)
        att_map = torch.matmul(att_map, self.att_weight)

        # apply temperature
        att_map = att_map / self.temp

        att_map = F.softmax(att_map, dim=-2)

        return att_map

    def _project(self, x, att_map):
        x1 = self.proj_with_att(torch.matmul(att_map.squeeze(-1), x))
        x2 = self.proj_without_att(x)

        return x1 + x2

    def _apply_BN(self, x):
        org_size = x.size()
        x = x.view(-1, org_size[-1])
        x = self.bn(x)
        x = x.view(org_size)

        return x

    def _init_new_params(self, *size):
        out = nn.Parameter(torch.FloatTensor(*size))
        nn.init.xavier_normal_(out)
        return out


class HtrgGraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, **kwargs):
        super().__init__()

        self.proj_type1 = nn.Linear(in_dim, in_dim)
        self.proj_type2 = nn.Linear(in_dim, in_dim)

        # attention map
        self.att_proj = nn.Linear(in_dim, out_dim)
        self.att_projM = nn.Linear(in_dim, out_dim)

        self.att_weight11 = self._init_new_params(out_dim, 1)
        self.att_weight22 = self._init_new_params(out_dim, 1)
        self.att_weight12 = self._init_new_params(out_dim, 1)
        self.att_weightM = self._init_new_params(out_dim, 1)

        # project
        self.proj_with_att = nn.Linear(in_dim, out_dim)
        self.proj_without_att = nn.Linear(in_dim, out_dim)

        self.proj_with_attM = nn.Linear(in_dim, out_dim)
        self.proj_without_attM = nn.Linear(in_dim, out_dim)

        # batch norm
        self.bn = nn.BatchNorm1d(out_dim)

        # dropout for inputs
        self.input_drop = nn.Dropout(p=0.2)

        # activate
        self.act = nn.SELU(inplace=True)

        # temperature
        self.temp = 1.
        if "temperature" in kwargs:
            self.temp = kwargs["temperature"]

    def forward(self, x1, x2, master=None):
        '''
        x1  :(#bs, #node, #dim)
        x2  :(#bs, #node, #dim)
        '''
        num_type1 = x1.size(1)
        num_type2 = x2.size(1)

        x1 = self.proj_type1(x1)
        x2 = self.proj_type2(x2)

        x = torch.cat([x1, x2], dim=1)

        if master is None:
            master = torch.mean(x, dim=1, keepdim=True)

        # apply input dropout
        x = self.input_drop(x)

        # derive attention map
        att_map = self._derive_att_map(x, num_type1, num_type2)

        # directional edge for master node
        master = self._update_master(x, master)

        # projection
        x = self._project(x, att_map)

        # apply batch norm
        x = self._apply_BN(x)
        x = self.act(x)

        x1 = x.narrow(1, 0, num_type1)
        x2 = x.narrow(1, num_type1, num_type2)

        return x1, x2, master

    def _update_master(self, x, master):

        att_map = self._derive_att_map_master(x, master)
        master = self._project_master(x, master, att_map)

        return master

    def _pairwise_mul_nodes(self, x):
        '''
        Calculates pairwise multiplication of nodes.
        - for attention map
        x           :(#bs, #node, #dim)
        out_shape   :(#bs, #node, #node, #dim)
        '''

        nb_nodes = x.size(1)
        x = x.unsqueeze(2).expand(-1, -1, nb_nodes, -1)
        x_mirror = x.transpose(1, 2)

        return x * x_mirror

    def _derive_att_map_master(self, x, master):
        '''
        x           :(#bs, #node, #dim)
        out_shape   :(#bs, #node, #node, 1)
        '''
        att_map = x * master
        att_map = torch.tanh(self.att_projM(att_map))

        att_map = torch.matmul(att_map, self.att_weightM)

        # apply temperature
        att_map = att_map / self.temp

        att_map = F.softmax(att_map, dim=-2)

        return att_map

    def _derive_att_map(self, x, num_type1, num_type2):
        '''
        x           :(#bs, #node, #dim)
        out_shape   :(#bs, #node, #node, 1)
        '''
        att_map = self._pairwise_mul_nodes(x)
        # size: (#bs, #node, #node, #dim_out)
        att_map = torch.tanh(self.att_proj(att_map))
        # size: (#bs, #node, #node, 1)

        att_board = torch.zeros_like(att_map[:, :, :, 0]).unsqueeze(-1)

        att_board[:, :num_type1, :num_type1, :] = torch.matmul(
            att_map[:, :num_type1, :num_type1, :], self.att_weight11)
        att_board[:, num_type1:, num_type1:, :] = torch.matmul(
            att_map[:, num_type1:, num_type1:, :], self.att_weight22)
        att_board[:, :num_type1, num_type1:, :] = torch.matmul(
            att_map[:, :num_type1, num_type1:, :], self.att_weight12)
        att_board[:, num_type1:, :num_type1, :] = torch.matmul(
            att_map[:, num_type1:, :num_type1, :], self.att_weight12)

        att_map = att_board

        # apply temperature
        att_map = att_map / self.temp

        att_map = F.softmax(att_map, dim=-2)

        return att_map

    def _project(self, x, att_map):
        x1 = self.proj_with_att(torch.matmul(att_map.squeeze(-1), x))
        x2 = self.proj_without_att(x)

        return x1 + x2

    def _project_master(self, x, master, att_map):

        x1 = self.proj_with_attM(torch.matmul(
            att_map.squeeze(-1).unsqueeze(1), x))
        x2 = self.proj_without_attM(master)

        return x1 + x2

    def _apply_BN(self, x):
        org_size = x.size()
        x = x.view(-1, org_size[-1])
        x = self.bn(x)
        x = x.view(org_size)

        return x

    def _init_new_params(self, *size):
        out = nn.Parameter(torch.FloatTensor(*size))
        nn.init.xavier_normal_(out)
        return out


class GraphPool(nn.Module):
    def __init__(self, k: float, in_dim: int, p: Union[float, int]):
        super().__init__()
        self.k = k
        self.sigmoid = nn.Sigmoid()
        self.proj = nn.Linear(in_dim, 1)
        self.drop = nn.Dropout(p=p) if p > 0 else nn.Identity()
        self.in_dim = in_dim

    def forward(self, h):
        Z = self.drop(h)
        weights = self.proj(Z)
        scores = self.sigmoid(weights)
        new_h = self.top_k_graph(scores, h, self.k)

        return new_h

    def top_k_graph(self, scores, h, k):
        _, n_nodes, n_feat = h.size()
        n_nodes = max(int(n_nodes * k), 1)
        _, idx = torch.topk(scores, n_nodes, dim=1)
        idx = idx.expand(-1, -1, n_feat)

        h = h * scores
        h = torch.gather(h, 1, idx)

        return h


class CONV(nn.Module):
    @staticmethod
    def to_mel(hz):
        return 2595 * np.log10(1 + hz / 700)

    @staticmethod
    def to_hz(mel):
        return 700 * (10**(mel / 2595) - 1)

    def __init__(self,
                 out_channels,
                 kernel_size,
                 sample_rate=16000,
                 in_channels=1,
                 stride=1,
                 padding=0,
                 dilation=1,
                 bias=False,
                 groups=1,
                 mask=False):
        super().__init__()
        if in_channels != 1:
            msg = "SincConv only support one input channel (here, in_channels = {%i})" % (in_channels)
            raise ValueError(msg)
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate

        # Forcing the filters to be odd (i.e, perfectly symmetrics)
        if kernel_size % 2 == 0:
            self.kernel_size = self.kernel_size + 1
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.mask = mask
        if bias:
            raise ValueError('SincConv does not support bias.')
        if groups > 1:
            raise ValueError('SincConv does not support groups.')

        NFFT = 512
        f = int(self.sample_rate / 2) * np.linspace(0, 1, int(NFFT / 2) + 1)
        fmel = self.to_mel(f)
        fmelmax = np.max(fmel)
        fmelmin = np.min(fmel)
        filbandwidthsmel = np.linspace(fmelmin, fmelmax, self.out_channels + 1)
        filbandwidthsf = self.to_hz(filbandwidthsmel)

        self.mel = filbandwidthsf
        self.hsupp = torch.arange(-(self.kernel_size - 1) / 2,
                                  (self.kernel_size - 1) / 2 + 1)
        self.band_pass = torch.zeros(self.out_channels, self.kernel_size)
        for i in range(len(self.mel) - 1):
            fmin = self.mel[i]
            fmax = self.mel[i + 1]
            hHigh = (2*fmax/self.sample_rate) * \
                np.sinc(2*fmax*self.hsupp/self.sample_rate)
            hLow = (2*fmin/self.sample_rate) * \
                np.sinc(2*fmin*self.hsupp/self.sample_rate)
            hideal = hHigh - hLow

            self.band_pass[i, :] = Tensor(np.hamming(
                self.kernel_size)) * Tensor(hideal)

    def forward(self, x, mask=False):
        band_pass_filter = self.band_pass.clone().to(x.device)
        if mask:
            A = np.random.uniform(0, 20)
            A = int(A)
            A0 = random.randint(0, band_pass_filter.shape[0] - A)
            band_pass_filter[A0:A0 + A, :] = 0
        else:
            band_pass_filter = band_pass_filter

        self.filters = (band_pass_filter).view(self.out_channels, 1,
                                               self.kernel_size)

        return F.conv1d(x,
                        self.filters,
                        stride=self.stride,
                        padding=self.padding,
                        dilation=self.dilation,
                        bias=None,
                        groups=1)


class Residual_block(nn.Module):
    def __init__(self, nb_filts, first=False):
        super().__init__()
        self.first = first

        if not self.first:
            self.bn1 = nn.BatchNorm2d(num_features=nb_filts[0])
        self.conv1 = nn.Conv2d(in_channels=nb_filts[0],
                               out_channels=nb_filts[1],
                               kernel_size=(2, 3),
                               padding=(1, 1),
                               stride=1)
        self.selu = nn.SELU(inplace=True)

        self.bn2 = nn.BatchNorm2d(num_features=nb_filts[1])
        self.conv2 = nn.Conv2d(in_channels=nb_filts[1],
                               out_channels=nb_filts[1],
                               kernel_size=(2, 3),
                               padding=(0, 1),
                               stride=1)

        if nb_filts[0] != nb_filts[1]:
            self.downsample = True
            self.conv_downsample = nn.Conv2d(in_channels=nb_filts[0],
                                             out_channels=nb_filts[1],
                                             padding=(0, 1),
                                             kernel_size=(1, 3),
                                             stride=1)

        else:
            self.downsample = False
        self.mp = nn.MaxPool2d((1, 3))

    def forward(self, x):
        identity = x
        if not self.first:
            out = self.bn1(x)
            out = self.selu(out)
        else:
            out = x
        out = self.conv1(x)

        out = self.bn2(out)
        out = self.selu(out)
        out = self.conv2(out)
        if self.downsample:
            identity = self.conv_downsample(identity)

        out += identity
        out = self.mp(out)
        return out


class AASISTModel(nn.Module):
    def __init__(self, d_args):
        super().__init__()

        self.d_args = d_args
        filts = d_args["filts"]
        gat_dims = d_args["gat_dims"]
        pool_ratios = d_args["pool_ratios"]
        temperatures = d_args["temperatures"]

        self.conv_time = CONV(out_channels=filts[0],
                              kernel_size=d_args["first_conv"],
                              in_channels=1)
        self.first_bn = nn.BatchNorm2d(num_features=1)

        self.drop = nn.Dropout(0.5, inplace=True)
        self.drop_way = nn.Dropout(0.2, inplace=True)
        self.selu = nn.SELU(inplace=True)

        self.encoder = nn.Sequential(
            nn.Sequential(Residual_block(nb_filts=filts[1], first=True)),
            nn.Sequential(Residual_block(nb_filts=filts[2])),
            nn.Sequential(Residual_block(nb_filts=filts[3])),
            nn.Sequential(Residual_block(nb_filts=filts[4])),
            nn.Sequential(Residual_block(nb_filts=filts[4])),
            nn.Sequential(Residual_block(nb_filts=filts[4])))

        self.pos_S = nn.Parameter(torch.randn(1, 23, filts[-1][-1]))
        self.master1 = nn.Parameter(torch.randn(1, 1, gat_dims[0]))
        self.master2 = nn.Parameter(torch.randn(1, 1, gat_dims[0]))

        self.GAT_layer_S = GraphAttentionLayer(filts[-1][-1],
                                               gat_dims[0],
                                               temperature=temperatures[0])
        self.GAT_layer_T = GraphAttentionLayer(filts[-1][-1],
                                               gat_dims[0],
                                               temperature=temperatures[1])

        self.HtrgGAT_layer_ST11 = HtrgGraphAttentionLayer(
            gat_dims[0], gat_dims[1], temperature=temperatures[2])
        self.HtrgGAT_layer_ST12 = HtrgGraphAttentionLayer(
            gat_dims[1], gat_dims[1], temperature=temperatures[2])

        self.HtrgGAT_layer_ST21 = HtrgGraphAttentionLayer(
            gat_dims[0], gat_dims[1], temperature=temperatures[2])

        self.HtrgGAT_layer_ST22 = HtrgGraphAttentionLayer(
            gat_dims[1], gat_dims[1], temperature=temperatures[2])

        self.pool_S = GraphPool(pool_ratios[0], gat_dims[0], 0.3)
        self.pool_T = GraphPool(pool_ratios[1], gat_dims[0], 0.3)
        self.pool_hS1 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.pool_hT1 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)

        self.pool_hS2 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.pool_hT2 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)

        self.out_layer = nn.Linear(5 * gat_dims[1], 2)

    def forward(self, x, Freq_aug=False):

        x = x.unsqueeze(1)
        x = self.conv_time(x, mask=Freq_aug)
        x = x.unsqueeze(dim=1)
        x = F.max_pool2d(torch.abs(x), (3, 3))
        x = self.first_bn(x)
        x = self.selu(x)

        e = self.encoder(x)

        e_S, _ = torch.max(torch.abs(e), dim=3)
        e_S = e_S.transpose(1, 2) + self.pos_S

        gat_S = self.GAT_layer_S(e_S)
        out_S = self.pool_S(gat_S)

        e_T, _ = torch.max(torch.abs(e), dim=2)
        e_T = e_T.transpose(1, 2)

        gat_T = self.GAT_layer_T(e_T)
        out_T = self.pool_T(gat_T)

        master1 = self.master1.expand(x.size(0), -1, -1)
        master2 = self.master2.expand(x.size(0), -1, -1)

        out_T1, out_S1, master1 = self.HtrgGAT_layer_ST11(
            out_T, out_S, master=self.master1)

        out_S1 = self.pool_hS1(out_S1)
        out_T1 = self.pool_hT1(out_T1)

        out_T_aug, out_S_aug, master_aug = self.HtrgGAT_layer_ST12(
            out_T1, out_S1, master=master1)
        out_T1 = out_T1 + out_T_aug
        out_S1 = out_S1 + out_S_aug
        master1 = master1 + master_aug

        out_T2, out_S2, master2 = self.HtrgGAT_layer_ST21(
            out_T, out_S, master=self.master2)
        out_S2 = self.pool_hS2(out_S2)
        out_T2 = self.pool_hT2(out_T2)

        out_T_aug, out_S_aug, master_aug = self.HtrgGAT_layer_ST22(
            out_T2, out_S2, master=master2)
        out_T2 = out_T2 + out_T_aug
        out_S2 = out_S2 + out_S_aug
        master2 = master2 + master_aug

        out_T1 = self.drop_way(out_T1)
        out_T2 = self.drop_way(out_T2)
        out_S1 = self.drop_way(out_S1)
        out_S2 = self.drop_way(out_S2)
        master1 = self.drop_way(master1)
        master2 = self.drop_way(master2)

        out_T = torch.max(out_T1, out_T2)
        out_S = torch.max(out_S1, out_S2)
        master = torch.max(master1, master2)

        T_max, _ = torch.max(torch.abs(out_T), dim=1)
        T_avg = torch.mean(out_T, dim=1)

        S_max, _ = torch.max(torch.abs(out_S), dim=1)
        S_avg = torch.mean(out_S, dim=1)

        last_hidden = torch.cat(
            [T_max, T_avg, S_max, S_avg, master.squeeze(1)], dim=1)

        last_hidden = self.drop(last_hidden)
        output = self.out_layer(last_hidden)

        return last_hidden, output

class AASISTDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        d_args = {
            "nb_samp": 64600,
            "first_conv": 128,
            "in_channels": 1,
            "filts": [70, [1, 32], [32, 32], [32, 64], [64, 64]],
            "gat_dims": [64, 32],
            "pool_ratios": [0.5, 0.7, 0.5, 0.5],
            "temperatures": [2.0, 2.0, 100.0]
        }
        self.model = AASISTModel(d_args)

        # Override out_layer if not strictly 2 classes.
        if num_classes != 2:
            self.model.out_layer = nn.Linear(5 * d_args["gat_dims"][1], num_classes)

    def forward(self, x):
        # x is (B, 1, T) or (B, T)
        if x.dim() == 3:
            x = x.squeeze(1) # Convert to (B, T)
        _, out = self.model(x)
        return out

# ============================================================
# 3. CQCC Baseline Detector (Acoustic Feature Baseline)
# ============================================================

class CQCCBaselineDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Input shape expected: (B, 1, 20, T)
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

# ============================================================
# 4. Custom Fusional Wav2Vec2 + CQCC with Cross-Attention + Graph
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=6000):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, dim))

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1)]

class BidirectionalCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.attn1 = nn.MultiheadAttention(dim, num_heads, batch_first=True, dropout=0.2)
        self.attn2 = nn.MultiheadAttention(dim, num_heads, batch_first=True, dropout=0.2)
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)

    def forward(self, x1, x2):
        # x1 attends to x2
        q1 = self.norm_q(x1)
        k2 = self.norm_kv(x2)
        v2 = k2
        out1, _ = self.attn1(q1, k2, v2)

        # x2 attends to x1
        q2 = self.norm_q(x2)
        k1 = self.norm_kv(x1)
        v1 = k1
        out2, _ = self.attn2(q2, k1, v1)
        return out1, out2

def align_sequences(x, target_len):
    """Linear interpolation to match sequence lengths"""
    x = x.transpose(1, 2)
    x = F.interpolate(x, size=target_len, mode='linear', align_corners=False)
    return x.transpose(1, 2)

class ImprovedWav2Vec2CQCCDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Wav2Vec2
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        # Freeze the Wav2Vec2 layer so it acts purely as a feature extractor
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        dim = self.wav2vec.config.hidden_size

        # CQCC encoder
        self.cqcc_conv = nn.Sequential(
            nn.Conv1d(20, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(dim),
            nn.GELU()
        )

        # Positional Encoding
        self.pos_enc = PositionalEncoding(dim)

        # Bidirectional Cross Attention
        self.cross_attn = BidirectionalCrossAttention(dim)

        # True Graph Transformer Backend (using GAT blocks from AASIST)
        self.graph_layers = nn.ModuleList([
            GraphBlock(dim) for _ in range(3)
        ])

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, wav, cqcc):
        if wav.dim() == 3:
            wav = wav.squeeze(1)

        # Wav2Vec2 features
        w2v = self.wav2vec(wav).last_hidden_state  # (B, T_w, D)

        # CQCC features
        if cqcc.dim() == 4:
            cqcc = cqcc.squeeze(1)
        cqcc_feat = self.cqcc_conv(cqcc).transpose(1, 2)  # (B, T_c, D)

        # Align lengths
        cqcc_feat = align_sequences(cqcc_feat, w2v.size(1))

        # Add positional encoding
        w2v = self.pos_enc(w2v)
        cqcc_feat = self.pos_enc(cqcc_feat)

        # Cross attention (bidirectional)
        f1, f2 = self.cross_attn(cqcc_feat, w2v)
        fused = f1 + f2

        # Graph Transformer processing on node sequences
        x = fused
        for layer in self.graph_layers:
            x = layer(x)

        # Global average pooling on the nodes
        pooled = x.mean(dim=1)

        return self.classifier(pooled)

# ============================================================
# 5. Ablation Models
# ============================================================

class AblationWav2Vec2GraphDetector(nn.Module):
    """Ablation 1: Wav2Vec2 only + Graph Backend (No CQCC, No Cross-Attention)"""
    def __init__(self, num_classes=2):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        dim = self.wav2vec.config.hidden_size
        self.pos_enc = PositionalEncoding(dim)

        self.graph_layers = nn.ModuleList([GraphBlock(dim) for _ in range(3)])
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, num_classes)
        )

    def forward(self, wav, cqcc=None): # Accept both but ignore CQCC
        if wav.dim() == 3:
            wav = wav.squeeze(1)

        w2v = self.wav2vec(wav).last_hidden_state
        w2v = self.pos_enc(w2v)

        x = w2v
        for layer in self.graph_layers:
            x = layer(x)

        pooled = x.mean(dim=1)
        return self.classifier(pooled)


class AblationCQCCGraphDetector(nn.Module):
    """Ablation 2: CQCC only + Graph Backend (No Wav2Vec2, No Cross-Attention)"""
    def __init__(self, num_classes=2):
        super().__init__()
        dim = 768 # Match Wav2Vec2 hidden size for fair comparison

        self.cqcc_conv = nn.Sequential(
            nn.Conv1d(20, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(dim),
            nn.GELU()
        )
        self.pos_enc = PositionalEncoding(dim)

        self.graph_layers = nn.ModuleList([GraphBlock(dim) for _ in range(3)])
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, num_classes)
        )

    def forward(self, cqcc):
        if cqcc.dim() == 4:
            cqcc = cqcc.squeeze(1)

        cqcc_feat = self.cqcc_conv(cqcc).transpose(1, 2)
        cqcc_feat = self.pos_enc(cqcc_feat)

        x = cqcc_feat
        for layer in self.graph_layers:
            x = layer(x)

        pooled = x.mean(dim=1)
        return self.classifier(pooled)


class AblationConcatGraphDetector(nn.Module):
    """Ablation 3: Wav2Vec2 + CQCC + Simple Concat Fusion + Graph Backend (No Cross-Attention)"""
    def __init__(self, num_classes=2):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        dim = self.wav2vec.config.hidden_size

        self.cqcc_conv = nn.Sequential(
            nn.Conv1d(20, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(dim),
            nn.GELU()
        )

        self.fusion_proj = nn.Linear(dim * 2, dim) # Project concatenated features back to dim
        self.pos_enc = PositionalEncoding(dim)

        self.graph_layers = nn.ModuleList([GraphBlock(dim) for _ in range(3)])
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, num_classes)
        )

    def forward(self, wav, cqcc):
        if wav.dim() == 3:
            wav = wav.squeeze(1)
        w2v = self.wav2vec(wav).last_hidden_state

        if cqcc.dim() == 4:
            cqcc = cqcc.squeeze(1)
        cqcc_feat = self.cqcc_conv(cqcc).transpose(1, 2)

        cqcc_feat = align_sequences(cqcc_feat, w2v.size(1))

        # Simple concat over feature dimension instead of cross-attention
        fused = torch.cat([w2v, cqcc_feat], dim=-1)
        fused = self.fusion_proj(fused)

        fused = self.pos_enc(fused)

        x = fused
        for layer in self.graph_layers:
            x = layer(x)

        pooled = x.mean(dim=1)
        return self.classifier(pooled)


class AblationCrossAttnLinearDetector(nn.Module):
    """Ablation 4: Wav2Vec2 + CQCC + Cross-Attention + Linear Backend (No Graph Transformer)"""
    def __init__(self, num_classes=2):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        dim = self.wav2vec.config.hidden_size

        self.cqcc_conv = nn.Sequential(
            nn.Conv1d(20, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(dim),
            nn.GELU()
        )

        self.pos_enc = PositionalEncoding(dim)
        self.cross_attn = BidirectionalCrossAttention(dim)

        # Richer MLP classifier since graph is missing
        self.classifier = nn.Sequential(
            nn.Linear(dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, wav, cqcc):
        if wav.dim() == 3:
            wav = wav.squeeze(1)
        w2v = self.wav2vec(wav).last_hidden_state

        if cqcc.dim() == 4:
            cqcc = cqcc.squeeze(1)
        cqcc_feat = self.cqcc_conv(cqcc).transpose(1, 2)

        cqcc_feat = align_sequences(cqcc_feat, w2v.size(1))

        w2v = self.pos_enc(w2v)
        cqcc_feat = self.pos_enc(cqcc_feat)

        f1, f2 = self.cross_attn(cqcc_feat, w2v)
        fused = f1 + f2

        # No graph layer, straight to global average pooling
        pooled = fused.mean(dim=1)
        return self.classifier(pooled)

## 2. Define Dataset and Data Utilities

In [3]:
import os
import hashlib
import torch
import torchaudio
import numpy as np
from torch.utils.data import Dataset
import librosa
from scipy.fftpack import dct

def compute_cqcc(wav_np, n_bins, sample_rate=16000, hop_length=160, num_coeffs=20):
    """Compute CQCC features from a mono waveform numpy array."""
    try:
        cqt = np.abs(
            librosa.cqt(
                wav_np,
                sr=sample_rate,
                n_bins=n_bins,
                hop_length=hop_length,
                fmin=librosa.note_to_hz('C1')
            )
        )
        log_power = librosa.amplitude_to_db(cqt, ref=np.max)
        cqcc = dct(log_power, type=2, axis=0, norm='ortho')[:num_coeffs]
        return torch.from_numpy(cqcc).unsqueeze(0).float()
    except Exception:
        # Fallback for very short or invalid audio.
        return torch.zeros((1, num_coeffs, 10), dtype=torch.float32)

class AudioDataset(Dataset):
    def __init__(self, data_dir=None, n_bins=60, augment=False, cqcc_cache_dir=None, target_lang=None):
        if data_dir is None:
            # Check if MLAAD-tiny exists, else fallback to 'data'
            # In Colab, we generally expect data_dir to be explicitly set,
            # but keeping this for robustness if the user runs the script locally.
            mlaad_dir = os.path.join(os.path.dirname(__file__), "..", "MLAAD-tiny")
            if os.path.exists(mlaad_dir):
                data_dir = mlaad_dir
            else:
                data_dir = os.path.join(os.path.dirname(__file__), "..", "data")

        self.data_dir = data_dir
        self.files = []
        self.labels = []
        self.n_bins = n_bins # Changed from n_mels
        self.augment = augment
        self.cqcc_cache_dir = cqcc_cache_dir
        self.target_lang = target_lang # Added back

        real_path = os.path.join(data_dir, "original")
        if not os.path.exists(real_path):
            real_path = os.path.join(data_dir, "real")

        fake_path = os.path.join(data_dir, "fake")

        # Collect all audio files and their labels
        for root, dirs, files in os.walk(real_path):
            dirs.sort()
            files.sort()
            for f in files:
                if f.endswith('.wav') or f.endswith('.flac'):
                    if self.target_lang:
                        rel_root = os.path.relpath(root, real_path).replace('\\', '/')
                        if not rel_root.startswith(self.target_lang):
                            continue
                    self.files.append(os.path.join(root, f))
                    self.labels.append(0) # 0 = Real

        for root, dirs, files in os.walk(fake_path):
            dirs.sort()
            files.sort()
            for f in files:
                if f.endswith('.wav') or f.endswith('.flac'):
                    if self.target_lang:
                        rel_root = os.path.relpath(root, fake_path).replace('\\', '/')
                        if not rel_root.startswith(self.target_lang):
                            continue
                    self.files.append(os.path.join(root, f))
                    self.labels.append(1) # 1 = Fake

        if self.cqcc_cache_dir is not None:
            os.makedirs(self.cqcc_cache_dir, exist_ok=True)

    def __len__(self):
        return len(self.files)

    def _cqcc_cache_path(self, audio_path):
        # Use relative path from data_dir to ensure unique caching based on source location
        rel_path = os.path.relpath(audio_path, start=self.data_dir)
        # Create a stable but unique hash for the filename to avoid issues with long/complex paths
        cache_key = hashlib.md5(audio_path.encode("utf-8")).hexdigest()
        rel_stem = os.path.splitext(rel_path)[0]
        # Replace directory separators with a safe character for filenames
        safe_name = rel_stem.replace(os.sep, "__")
        return os.path.join(self.cqcc_cache_dir, f"{safe_name}_{cache_key}.pt")

    def _load_or_compute_cqcc(self, audio_path, wav_np, bypass_cache=False):
        if self.cqcc_cache_dir is None or bypass_cache:
            return compute_cqcc(wav_np, n_bins=self.n_bins)

        cache_path = self._cqcc_cache_path(audio_path)
        if os.path.exists(cache_path):
            return torch.load(cache_path, map_location="cpu")

        cqcc = compute_cqcc(wav_np, n_bins=self.n_bins)
        torch.save(cqcc, cache_path)
        return cqcc

    def precompute_cqcc_cache(self, force=False):
        """Materialize CQCC features to disk so training can reuse them."""
        import tqdm
        if self.cqcc_cache_dir is None:
            raise ValueError("cqcc_cache_dir must be set to precompute CQCC features.")

        try:
            from tqdm.notebook import tqdm
            iterable_files = tqdm(self.files, desc="Precomputing CQCC Cache")
        except ImportError:
            iterable_files = self.files

        total = len(self.files)
        for idx, audio_path in enumerate(iterable_files):
            cache_path = self._cqcc_cache_path(audio_path)
            if not force and os.path.exists(cache_path):
                continue

            try:
                wav_np, _ = librosa.load(audio_path, sr=16000, mono=True)
                cqcc = compute_cqcc(wav_np, n_bins=self.n_bins)
                torch.save(cqcc, cache_path)
            except Exception as e:
                print(f"Error precomputing CQCC for {audio_path}: {e}")

            if (idx + 1) % 100 == 0 or idx + 1 == total:
                print(f"Precomputed CQCC {idx + 1}/{total}")


    def __getitem__(self, idx):
        audio_path = self.files[idx]
        wav_np, sr = librosa.load(audio_path, sr=16000, mono=True)

        is_augmented = False
        # Augmentation on raw audio (Data Augmentation for generalizability)
        if self.augment and np.random.rand() < 0.3:
            # Apply only ONE augmentation type per sample to avoid over-modification
            aug_type = np.random.choice(['noise', 'speed', 'pitch'], p=[0.33, 0.33, 0.34])

            if aug_type == 'noise':
                # SNR-based noise addition (reverted to original robust method)
                signal_power = np.mean(wav_np**2)
                if signal_power > 1e-10:
                    snr_db = np.random.uniform(10, 30)
                    snr_linear = 10**(snr_db / 10)
                    noise_power = signal_power / snr_linear
                    noise = np.random.randn(len(wav_np)) * np.sqrt(noise_power)
                    wav_np = wav_np + noise
                is_augmented = True
            elif aug_type == 'speed':
                # Mild speed perturbation
                speed_factor = np.random.uniform(0.95, 1.05)
                wav_np = librosa.effects.time_stretch(wav_np, rate=speed_factor)
                is_augmented = True
            elif aug_type == 'pitch':
                # Subtle pitch shift
                n_steps = np.random.uniform(-1, 1)
                wav_np = librosa.effects.pitch_shift(wav_np, sr=sr, n_steps=n_steps)
                is_augmented = True

        # Crop or pad to exactly 64600 samples (AASIST standard)
        target_len = 64600
        if len(wav_np) > target_len:
            # Center crop or random crop for augment instead of taking just the start.
            if self.augment:
                start = np.random.randint(0, len(wav_np) - target_len)
            else:
                start = (len(wav_np) - target_len) // 2
            wav_np = wav_np[start:start+target_len]
        elif len(wav_np) < target_len:
            pad = target_len - len(wav_np)
            wav_np = np.pad(wav_np, (0, pad), 'constant')

        wav = torch.from_numpy(wav_np).unsqueeze(0).float()

        cqcc = self._load_or_compute_cqcc(audio_path, wav_np, bypass_cache=is_augmented)

        return wav, cqcc, self.labels[idx]


def collate_variable_length(batch):

    wavs, cqccs, labels = zip(*batch)
    labels = torch.tensor(labels)

    # ---------- WAVE ----------
    max_wav_len = max(w.shape[-1] for w in wavs)

    wavs_padded = []
    for w in wavs:
        if w.shape[-1] < max_wav_len:
            pad = max_wav_len - w.shape[-1]
            w = torch.nn.functional.pad(w, (0, pad))
        wavs_padded.append(w)

    wavs = torch.stack(wavs_padded, dim=0)

    # ---------- CQCC ----------
    max_cqcc_len = max(c.shape[-1] for c in cqccs)

    cqccs_padded = []
    for c in cqccs:
        if c.shape[-1] < max_cqcc_len:
            # CQCC tensor has shape (1, num_coeffs, T), pad last dimension
            pad = max_cqcc_len - c.shape[-1]
            c = torch.nn.functional.pad(c, (0, pad))
        cqccs_padded.append(c)

    cqccs = torch.stack(cqccs_padded, dim=0) # Result: (B, 1, num_coeffs, max_cqcc_len)

    return wavs, cqccs, labels

In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np
import random
from tqdm.notebook import tqdm # Import tqdm for progress bar


def train_model(model, train_dataloader, criterion, optimizer, epochs=5, input_type='wav', device=None, val_dataloader=None, eval_interval=1, patience=2, model_save_path=None):

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.to(device)

    loss_history = []
    best_val_metric = float('inf') # For min_dcf, lower is better
    patience_counter = 0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0
        # Wrap the dataloader with tqdm for a progress bar
        for batch_idx, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} - Training")):

            wavs, cqccs, labels = batch
            wavs = wavs.to(device)
            cqccs = cqccs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # 🔥 Clean input handling
            if input_type == 'wav':
                outputs = model(wavs)

            elif input_type == 'cqcc':
                outputs = model(cqccs)

            elif input_type == 'wav_and_cqcc':
                outputs = model(wavs, cqccs)

            else:
                raise ValueError("invalid input_type (mel removed)")

            loss = criterion(outputs, labels)

            # Scale the loss and call backward()
            loss.backward()
            # Unscale gradients and call optimizer.step()
            optimizer.step()
            # Removed scaler update

            epoch_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1);

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Print intermediate progress within the epoch
            if batch_idx % 500 == 0 and batch_idx > 0: # Report every 500 batches
                current_acc = 100 * correct / total
                current_loss = epoch_loss / (batch_idx + 1)
                print(f"  Batch {batch_idx}/{len(train_dataloader)} | Loss: {current_loss:.4f} | Acc: {current_acc:.2f}%")

        acc = 100 * correct / total if total > 0 else 0
        avg_loss = epoch_loss / len(train_dataloader)
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} | Training Loss: {avg_loss:.4f} | Training Acc: {acc:.2f}%")

        # Validation and Early Stopping
        if val_dataloader is not None and (epoch + 1) % eval_interval == 0:
            print(f"Epoch {epoch+1}/{epochs} - Evaluating on Validation Set...")
            _, _, _, val_eer, val_min_dcf, val_accuracy = evaluate_model(
                model, val_dataloader, input_type=input_type, device=device
            )
            print(f"  Validation | EER={val_eer*100:.2f}% | minDCF={val_min_dcf:.4f} | Accuracy={val_accuracy:.2f}")

            if val_min_dcf < best_val_metric:
                best_val_metric = val_min_dcf
                patience_counter = 0
                best_epoch = epoch + 1
                if model_save_path:
                    torch.save(model.state_dict(), model_save_path)
                    print(f"  Saved best model to {model_save_path} (minDCF: {best_val_metric:.4f})")
            else:
                patience_counter += 1
                print(f"  Validation minDCF did not improve. Patience: {patience_counter}/{patience}")

                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch+1} epochs. Best minDCF: {best_val_metric:.4f} at epoch {best_epoch}")
                    if model_save_path:
                        print(f"Loading best model from {model_save_path}")
                        model.load_state_dict(torch.load(model_save_path))
                    return loss_history # Stop training

    return loss_history


def evaluate_model(model, dataloader, input_type='mel', device=None):

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in tqdm(dataloader, desc="Evaluating"):

            wavs, cqccs, labels = batch
            wavs = wavs.to(device)
            cqccs = cqccs.to(device)
            labels = labels.to(device)

            # Removed autocast for mixed precision during evaluation
            if input_type == 'wav':
                outputs = model(wavs)

            elif input_type == 'cqcc':
                outputs = model(cqccs)

            elif input_type == 'wav_and_cqcc':
                outputs = model(wavs, cqccs)

            # The following line is redundant as labels are already extracted and moved to device above.
            # labels = batch[-1].to(device)

            probs = torch.softmax(outputs, dim=1)[:, 1]

            all_labels.extend(labels.tolist())
            all_probs.extend(probs.tolist())

    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    # ------------------
    # EER (Equal Error Rate)
    # ------------------
    fnr = 1 - tpr
    eer_index = np.nanargmin(np.absolute(fnr - fpr))
    eer = fpr[eer_index]

    # ------------------
    # minDCF (Minimum Detection Cost Function)
    # Parameters according to ASVspoof 5 Evaluation Plan (Track 1)
    # ------------------
    P_spoof = 0.05      # Prior probability of a spoofing attack (\pi_{spf})
    P_bonafide = 0.95   # Prior probability of a real/bonafide utterance (1 - \pi_{spf})
    C_miss = 1          # Cost of falsely rejecting a real voice (Miss)
    C_fa = 10           # Cost of falsely accepting a spoof (False Alarm)

    # In the dataset, 0 = real (bonafide), 1 = fake (spoof)
    # fpr (False Positive Rate) = predicted fake (1) when true is real (0). This is a "miss" in ASVspoof.
    # fnr (False Negative Rate) = predicted real (0) when true is fake (1). This is a "false alarm" in ASVspoof.
    P_miss = fpr
    P_fa = fnr

    # Raw DCF = C_miss * P_bonafide * P_miss + C_fa * P_spoof * P_fa
    # Normalized by the default DCF (min cost of predicting all bonafide vs all spoof)
    dcf_default = min(C_miss * P_bonafide, C_fa * P_spoof)
    dcf_array = (C_miss * P_bonafide * P_miss + C_fa * P_spoof * P_fa) / dcf_default
    min_dcf = np.min(dcf_array)

    # Overall Accuracy (using 0.5 threshold)
    preds = [1 if p > 0.5 else 0 for p in all_probs]
    correct = sum(1 for p, l in zip(preds, all_labels) if p == l)
    accuracy = correct / len(all_labels) if len(all_labels) > 0 else 0

    return fpr, tpr, roc_auc, eer, min_dcf, accuracy

# Set up and seed

In [5]:
import os, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# For reproducible data loading and splitting
g_dataloader = torch.Generator()
g_dataloader.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Dataset + Loader

In [6]:
import os
import torch
from torch.utils.data import DataLoader, Subset
import random
import numpy as np # Ensure numpy is imported

# Define paths based on the notebook structure
DATA_ROOT = "/content/drive/MyDrive/MLAAD-tiny"
CQCC_CACHE_DIR = os.path.join(DATA_ROOT, "precomputed_features", "cqcc")

# 1. Initialize the AudioDataset (this also discovers all audio files)
# This will now initialize datasets for specific languages for train/test as per train.py

# 2. Precompute CQCC features (acts like preprocess_cqcc.py)
# This will be done in the next cell after defining full_train_dataset and full_test_dataset


In [8]:
import os
import torch
from torch.utils.data import DataLoader, Subset
import random
import numpy as np # Ensure numpy is imported

# Define validation split ratio
VAL_SPLIT_RATIO = 0.2

# Define paths based on the notebook structure
DATA_ROOT = "/content/drive/MyDrive/MLAAD-tiny"
CQCC_CACHE_DIR = os.path.join(DATA_ROOT, "precomputed_features", "cqcc")

# 1. Initialize the AudioDataset (this also discovers all audio files)
# This will now initialize datasets for specific languages for train/test as per train.py

# Load English Dataset for Training and Validation
print("Loading English Dataset for training/validation...")
full_en_dataset = AudioDataset(data_dir=DATA_ROOT, augment=False, cqcc_cache_dir=CQCC_CACHE_DIR, target_lang="en")
total_en = len(full_en_dataset)
if total_en == 0:
    raise ValueError("No English data found for target_lang='en'. Check data_dir and directory layout.")

# Create train/validation split
train_size = int((1.0 - VAL_SPLIT_RATIO) * total_en)
val_size = total_en - train_size
indices = torch.randperm(total_en, generator=g_dataloader).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(
    AudioDataset(data_dir=DATA_ROOT, augment=True, cqcc_cache_dir=CQCC_CACHE_DIR, target_lang="en"),
    train_indices
)
val_dataset = Subset(
    AudioDataset(data_dir=DATA_ROOT, augment=False, cqcc_cache_dir=CQCC_CACHE_DIR, target_lang="en"),
    val_indices
)

# Load German Dataset for Testing
print("Loading German Dataset for Testing...")
test_dataset = AudioDataset(data_dir=DATA_ROOT, augment=False, cqcc_cache_dir=CQCC_CACHE_DIR, target_lang="de")

print(f"Discovered {len(train_dataset)} English audio files for training.")
print(f"Discovered {len(val_dataset)} English audio files for validation.")
print(f"Discovered {len(test_dataset)} German audio files for testing.")

# Precompute CQCC features for all datasets (if not already cached)
print("Precomputing CQCC cache for Training Data (English)...")
AudioDataset(data_dir=DATA_ROOT, augment=False, cqcc_cache_dir=CQCC_CACHE_DIR, target_lang="en").precompute_cqcc_cache(force=False)
print("Precomputing CQCC cache for Validation Data (English)...")
AudioDataset(data_dir=DATA_ROOT, augment=False, cqcc_cache_dir=CQCC_CACHE_DIR, target_lang="en").precompute_cqcc_cache(force=False)
print("Precomputing CQCC cache for Testing Data (German)...")
test_dataset.precompute_cqcc_cache(force=False)

# 4. Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_variable_length,
    num_workers=2,
    pin_memory=True,
    generator=g_dataloader,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_variable_length,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_variable_length,
    num_workers=2,
    pin_memory=True
)

models_dir = "/content/drive/MyDrive/models"
os.makedirs(models_dir, exist_ok=True)

criterion = nn.CrossEntropyLoss()

Loading English Dataset for training/validation...
Loading German Dataset for Testing...
Discovered 9976 English audio files for training.
Discovered 2494 English audio files for validation.
Discovered 2832 German audio files for testing.
Precomputing CQCC cache for Training Data (English)...


Precomputing CQCC Cache:   0%|          | 0/12470 [00:00<?, ?it/s]

Precomputing CQCC cache for Validation Data (English)...


Precomputing CQCC Cache:   0%|          | 0/12470 [00:00<?, ?it/s]

Precomputing CQCC cache for Testing Data (German)...


Precomputing CQCC Cache:   0%|          | 0/2832 [00:00<?, ?it/s]

# TRAIN EACH MODEL IN SEPARATE CELLS

In [10]:
def load_model(model_class, path, check_exists=True):
    if check_exists and not os.path.exists(path):
        print(f"Warning: Model file not found at {path}. Skipping evaluation for this model.")
        return None
    model = model_class(num_classes=2).to(device)
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

In [ ]:
print("\n--- Training Wav2Vec2 Baseline ---")

wav2vec_model = Wav2Vec2SpoofDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(wav2vec_model.parameters(), lr=1e-4)

wav2vec_model_save_path = os.path.join(models_dir, "wav2vec2.pth")

train_model(
    wav2vec_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='wav',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=wav2vec_model_save_path
)

# The best model will already be loaded by train_model if early stopping was triggered
# Otherwise, the last epoch's model is saved.

print("\n--- Evaluating Wav2Vec2 Model on Train and Test Sets ---")

wav2vec2_model_loaded = load_model(Wav2Vec2SpoofDetector, wav2vec_model_save_path, check_exists=True)

if wav2vec2_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        wav2vec2_model_loaded,
        train_loader,
        input_type='wav',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        wav2vec2_model_loaded,
        test_loader,
        input_type='wav',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("Wav2Vec2 model could not be loaded or was not trained.")


--- Training Wav2Vec2 Baseline ---


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/5 - Training:   0%|          | 0/1247 [00:00<?, ?it/s]

  Batch 500/1247 | Loss: 0.6145 | Acc: 64.90%
  Batch 1000/1247 | Loss: 0.5437 | Acc: 71.13%
Epoch 1/5 | Training Loss: 0.5202 | Training Acc: 72.83%
Epoch 1/5 - Evaluating on Validation Set...


...


Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.942 | EER=13.70% | minDCF=0.3451 | Accuracy=0.86

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.769 | EER=30.15% | minDCF=0.6982 | Accuracy=0.61


In [ ]:
print("\n--- Training AASIST Baseline ---")

aasist_model = AASISTDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(aasist_model.parameters(), lr=1e-4)

aasist_model_save_path = os.path.join(models_dir, "aasist.pth")

train_model(
    aasist_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='wav',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=aasist_model_save_path
)

print("\n--- Evaluating AASIST Model on Train and Test Sets ---")

aasist_model_loaded = load_model(AASISTDetector, aasist_model_save_path, check_exists=True)

if aasist_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        aasist_model_loaded,
        train_loader,
        input_type='wav',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        aasist_model_loaded,
        test_loader,
        input_type='wav',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("AASIST model could not be loaded or was not trained.")


--- Training AASIST Baseline ---


Epoch 1/5 - Training:   0%|          | 0/1247 [00:00<?, ?it/s]

  Batch 500/1247 | Loss: 0.7594 | Acc: 59.21%
  Batch 1000/1247 | Loss: 0.7140 | Acc: 60.90%
Epoch 1/5 | Training Loss: 0.6953 | Training Acc: 61.83%
Epoch 1/5 - Evaluating on Validation Set...


Evaluating:   0%|          | 0/312 [00:00<?, ?it/s]

  Validation | EER=23.29% | minDCF=0.5868 | Accuracy=0.76
  Saved best model to /content/drive/MyDrive/models/aasist.pth (minDCF: 0.5868)


...

  Validation | EER=11.02% | minDCF=0.2617 | Accuracy=0.84
  Saved best model to /content/drive/MyDrive/models/aasist.pth (minDCF: 0.2617)

--- Evaluating AASIST Model on Train and Test Sets ---

Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.943 | EER=13.12% | minDCF=0.3249 | Accuracy=0.82

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.860 | EER=22.12% | minDCF=0.5666 | Accuracy=0.64


In [ ]:
print("\n--- Training CQCC Baseline ---")

cqcc_model = CQCCBaselineDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(cqcc_model.parameters(), lr=1e-4)

cqcc_model_save_path = os.path.join(models_dir, "cqcc_baseline.pth")

train_model(
    cqcc_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='cqcc',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=cqcc_model_save_path
)

print("\n--- Evaluating CQCC Model on Train and Test Sets ---")

cqcc_model_loaded = load_model(CQCCBaselineDetector, cqcc_model_save_path, check_exists=True)

if cqcc_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        cqcc_model_loaded,
        train_loader,
        input_type='cqcc',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        cqcc_model_loaded,
        test_loader,
        input_type='cqcc',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("CQCC model could not be loaded or was not trained.")


--- Training CQCC Baseline ---


Epoch 1/5 - Training:   0%|          | 0/1247 [00:00<?, ?it/s]

  Batch 500/1247 | Loss: 0.6649 | Acc: 59.96%
  Batch 1000/1247 | Loss: 0.6484 | Acc: 62.61%
Epoch 1/5 | Training Loss: 0.6430 | Training Acc: 63.55%
Epoch 1/5 - Evaluating on Validation Set...


Evaluating:   0%|          | 0/312 [00:00<?, ?it/s]

  Validation | EER=30.13% | minDCF=0.6309 | Accuracy=0.69
  Saved best model to /content/drive/MyDrive/models/cqcc_baseline.pth (minDCF: 0.6309)


...

  Validation | EER=27.38% | minDCF=0.5626 | Accuracy=0.64
  Validation minDCF did not improve. Patience: 2/2
Early stopping triggered after 4 epochs. Best minDCF: 0.4894 at epoch 2
Loading best model from /content/drive/MyDrive/models/cqcc_baseline.pth

--- Evaluating CQCC Model on Train and Test Sets ---

Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.804 | EER=26.03% | minDCF=0.5948 | Accuracy=0.75

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.642 | EER=40.38% | minDCF=0.9311 | Accuracy=0.60


In [ ]:
print("\n--- Training Custom Fusion Detector ---")

custom_model = ImprovedWav2Vec2CQCCDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(custom_model.parameters(), lr=1e-4)

custom_model_save_path = os.path.join(models_dir, "custom_hybrid.pth")

train_model(
    custom_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='wav_and_cqcc',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=custom_model_save_path
)

print("\n--- Evaluating Custom Fusion Detector on Train and Test Sets ---")

custom_model_loaded = load_model(ImprovedWav2Vec2CQCCDetector, custom_model_save_path, check_exists=True)

if custom_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        custom_model_loaded,
        train_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        custom_model_loaded,
        test_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("Custom Fusion model could not be loaded or was not trained.")


--- Training Custom Fusion Detector ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

...


Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.976 | EER=8.30% | minDCF=0.1983 | Accuracy=0.92

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.882 | EER=17.80% | minDCF=0.5080 | Accuracy=0.83


In [ ]:
print("\n--- Training Ablation Wav2Vec2GraphDetector ---")

ablation_w2v_graph_model = AblationWav2Vec2GraphDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(ablation_w2v_graph_model.parameters(), lr=1e-4)

ablation_w2v_graph_model_save_path = os.path.join(models_dir, "ablation_w2v_graph.pth")

train_model(
    ablation_w2v_graph_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='wav',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=ablation_w2v_graph_model_save_path
)

print("\n--- Evaluating Ablation Wav2Vec2GraphDetector on Train and Test Sets ---")

ablation_w2v_graph_model_loaded = load_model(AblationWav2Vec2GraphDetector, ablation_w2v_graph_model_save_path, check_exists=True)

if ablation_w2v_graph_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        ablation_w2v_graph_model_loaded,
        train_loader,
        input_type='wav',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        ablation_w2v_graph_model_loaded,
        test_loader,
        input_type='wav',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("Ablation Wav2Vec2Graph model could not be loaded or was not trained.")


--- Training Ablation Wav2Vec2GraphDetector ---


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/5 - Training:   0%|          | 0/1247 [00:00<?, ?it/s]

  Batch 500/1247 | Loss: 0.5784 | Acc: 67.51%
  Batch 1000/1247 | Loss: 0.5134 | Acc: 72.59%
Epoch 1/5 | Training Loss: 0.4950 | Training Acc: 73.84%
Epoch 1/5 - Evaluating on Validation Set...


...


Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.956 | EER=11.70% | minDCF=0.2930 | Accuracy=0.88

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.779 | EER=28.86% | minDCF=0.7492 | Accuracy=0.63


In [ ]:
print("\n--- Training Ablation CQCCGraphDetector ---")

ablation_cqcc_graph_model = AblationCQCCGraphDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(ablation_cqcc_graph_model.parameters(), lr=1e-4)

ablation_cqcc_graph_model_save_path = os.path.join(models_dir, "ablation_cqcc_graph.pth")

train_model(
    ablation_cqcc_graph_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='cqcc',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=ablation_cqcc_graph_model_save_path
)

print("\n--- Evaluating Ablation CQCCGraphDetector on Train and Test Sets ---")

ablation_cqcc_graph_model_loaded = load_model(AblationCQCCGraphDetector, ablation_cqcc_graph_model_save_path, check_exists=True)

if ablation_cqcc_graph_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        ablation_cqcc_graph_model_loaded,
        train_loader,
        input_type='cqcc',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        ablation_cqcc_graph_model_loaded,
        test_loader,
        input_type='cqcc',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("Ablation CQCCGraph model could not be loaded or was not trained.")


--- Training Ablation CQCCGraphDetector ---


Epoch 1/5 - Training:   0%|          | 0/1247 [00:00<?, ?it/s]

  Batch 500/1247 | Loss: 0.4818 | Acc: 76.20%
  Batch 1000/1247 | Loss: 0.4189 | Acc: 80.26%
Epoch 1/5 | Training Loss: 0.4014 | Training Acc: 81.08%
Epoch 1/5 - Evaluating on Validation Set...


Evaluating:   0%|          | 0/312 [00:00<?, ?it/s]

  Validation | EER=5.26% | minDCF=0.0912 | Accuracy=0.93
  Saved best model to /content/drive/MyDrive/models/ablation_cqcc_graph.pth (minDCF: 0.0912)


...

  Validation | EER=1.50% | minDCF=0.0217 | Accuracy=0.97
  Saved best model to /content/drive/MyDrive/models/ablation_cqcc_graph.pth (minDCF: 0.0217)

--- Evaluating Ablation CQCCGraphDetector on Train and Test Sets ---

Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.980 | EER=8.23% | minDCF=0.1887 | Accuracy=0.91

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.890 | EER=16.06% | minDCF=0.4219 | Accuracy=0.86


In [ ]:
print("\n--- Training Ablation ConcatGraphDetector ---")

ablation_concat_graph_model = AblationConcatGraphDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(ablation_concat_graph_model.parameters(), lr=1e-4)

ablation_concat_graph_model_save_path = os.path.join(models_dir, "ablation_concat_graph.pth")

train_model(
    ablation_concat_graph_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='wav_and_cqcc',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=ablation_concat_graph_model_save_path
)

print("\n--- Evaluating Ablation ConcatGraphDetector on Train and Test Sets ---")

ablation_concat_graph_model_loaded = load_model(AblationConcatGraphDetector, ablation_concat_graph_model_save_path, check_exists=True)

if ablation_concat_graph_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        ablation_concat_graph_model_loaded,
        train_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        ablation_concat_graph_model_loaded,
        test_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("Ablation ConcatGraph model could not be loaded or was not trained.")


--- Training Ablation ConcatGraphDetector ---


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/5 - Training:   0%|          | 0/1247 [00:00<?, ?it/s]

  Batch 500/1247 | Loss: 0.4798 | Acc: 75.60%
  Batch 1000/1247 | Loss: 0.4232 | Acc: 79.02%
Epoch 1/5 | Training Loss: 0.4022 | Training Acc: 80.51%
Epoch 1/5 - Evaluating on Validation Set...


...


Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.974 | EER=8.42% | minDCF=0.2005 | Accuracy=0.92

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.859 | EER=20.98% | minDCF=0.5491 | Accuracy=0.69


In [9]:
print("\n--- Training Ablation CrossAttnLinearDetector ---")

ablation_cross_attn_linear_model = AblationCrossAttnLinearDetector(num_classes=2).to(device)
optimizer = torch.optim.Adam(ablation_cross_attn_linear_model.parameters(), lr=1e-4)

ablation_cross_attn_linear_model_save_path = os.path.join(models_dir, "ablation_cross_attn_linear.pth")

train_model(
    ablation_cross_attn_linear_model,
    train_loader,
    criterion,
    optimizer,
    epochs=5,
    input_type='wav_and_cqcc',
    device=device,
    val_dataloader=val_loader,
    eval_interval=1,
    patience=2, # Early stopping patience
    model_save_path=ablation_cross_attn_linear_model_save_path
)

print("\n--- Evaluating Ablation CrossAttnLinearDetector on Train and Test Sets ---")

ablation_cross_attn_linear_model_loaded = load_model(AblationCrossAttnLinearDetector, ablation_cross_attn_linear_model_save_path, check_exists=True)

if ablation_cross_attn_linear_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        ablation_cross_attn_linear_model_loaded,
        train_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        ablation_cross_attn_linear_model_loaded,
        test_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("Ablation CrossAttnLinear model could not be loaded or was not trained.")


--- Training Ablation CrossAttnLinearDetector ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

Epoch 5/5 - Training:   0%|          | 0/1247 [00:00<?, ?it/s]

  Batch 500/1247 | Loss: 0.1897 | Acc: 91.87%
  Batch 1000/1247 | Loss: 0.1975 | Acc: 91.30%
Epoch 5/5 | Training Loss: 0.1967 | Training Acc: 91.45%
Epoch 5/5 - Evaluating on Validation Set...


Evaluating:   0%|          | 0/312 [00:00<?, ?it/s]

  Validation | EER=2.73% | minDCF=0.0563 | Accuracy=0.97
  Saved best model to /content/drive/MyDrive/models/ablation_cross_attn_linear.pth (minDCF: 0.0563)

--- Evaluating Ablation CrossAttnLinearDetector on Train and Test Sets ---


NameError: name 'load_model' is not defined

In [11]:
ablation_cross_attn_linear_model_loaded = load_model(AblationCrossAttnLinearDetector, ablation_cross_attn_linear_model_save_path, check_exists=True)

if ablation_cross_attn_linear_model_loaded is not None:
    print("\nEvaluation on Training Set (English):")
    _, _, auc_train, eer_train, min_dcf_train, acc_train = evaluate_model(
        ablation_cross_attn_linear_model_loaded,
        train_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Train Set | AUC={auc_train:.3f} | EER={eer_train*100:.2f}% | minDCF={min_dcf_train:.4f} | Accuracy={acc_train:.2f}")

    print("\nEvaluation on Testing Set (German):")
    _, _, auc_test, eer_test, min_dcf_test, acc_test = evaluate_model(
        ablation_cross_attn_linear_model_loaded,
        test_loader,
        input_type='wav_and_cqcc',
        device=device
    )
    print(f"Test Set | AUC={auc_test:.3f} | EER={eer_test*100:.2f}% | minDCF={min_dcf_test:.4f} | Accuracy={acc_test:.2f}")
else:
    print("Ablation CrossAttnLinear model could not be loaded or was not trained.")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Evaluation on Training Set (English):


Evaluating:   0%|          | 0/1247 [00:00<?, ?it/s]

Train Set | AUC=0.982 | EER=6.76% | minDCF=0.1535 | Accuracy=0.92

Evaluation on Testing Set (German):


Evaluating:   0%|          | 0/354 [00:00<?, ?it/s]

Test Set | AUC=0.883 | EER=18.94% | minDCF=0.5252 | Accuracy=0.81


In [ ]:
import torch
torch.cuda.empty_cache()
print("CUDA cache cleared.")

CUDA cache cleared.
